# Tanager Mangrove Mapping - 00 EMIT Data Inspector

| | |
|---|---|
| **Authors**     | Muhammad Wahyu Ramadhan, Athar Abdurrahman B., Diniyarti |
| **Competition** | Planet Tanager Open Data Competition 2026 |
| **Topic**       | Transferable Mangrove Extent and Biomass Mapping Using Adaptive Spectral Thresholds |
| **Date**        | June 2026 |

---

**Scope:** Exploratory inspection of EMIT masked GeoTIFF files — metadata summary, spectral profile visualization, single-band spatial preview, and RGB composite.


## 0. Environment Setup

In [ ]:
# !pip install rasterio matplotlib numpy


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import glob
import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import rasterio
from src.spatial_viz import reproject_raster_to_4326, reproject_rgb_composite_to_4326, format_map_axes

# ============================================================
# Project root
# ============================================================
# Google Colab (Google Drive mounted)
ROOT      = Path('/content/drive/MyDrive/PROJECT/Planet Tanager Competition 2026/tanager-mangrove-mapping')

# Local (uncomment if running locally)
# ROOT = Path('..').resolve()

EMIT_DIR  = ROOT / 'data' / 'emit' / 'masked'

print(f'ROOT     : {ROOT}')
print(f'EMIT_DIR : {EMIT_DIR}')


## 1. File Discovery

In [ ]:
tif_files = sorted(glob.glob(str(EMIT_DIR / '*.tif')))

if not tif_files:
    print(f'[ERROR] No .tif files found in {EMIT_DIR}')
else:
    print(f'Found {len(tif_files)} EMIT file(s):')
    for p in tif_files:
        print(f'  {os.path.basename(p)}')


## 2. Metadata Inspection

In [ ]:
print('=== METADATA SUMMARY ===')
for i, file_path in enumerate(tif_files, start=1):
    filename = os.path.basename(file_path)
    with rasterio.open(file_path) as src:
        print(f'{i}. {filename}')
        print(f'   Dimensions : {src.width} x {src.height} px')
        print(f'   Bands      : {src.count}')
        print(f'   CRS        : {src.crs}')
        print()


## 3. Spectral Profile

Dual-panel plot: full-range view (top) and zoom on bands 1–285 (bottom), sampled at the scene centre pixel.


In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10),
                               gridspec_kw={'height_ratios': [1, 1.5]})
colors = plt.cm.tab10.colors

max_y_zoom = -np.inf
min_y_zoom =  np.inf

for i, file_path in enumerate(tif_files):
    filename = os.path.basename(file_path)

    with rasterio.open(file_path) as src:
        row_y = src.height // 2
        col_x = src.width  // 2
        profile = list(src.sample([src.xy(row_y, col_x)]))[0].astype('float32')

        if src.nodata is not None:
            profile = np.where(profile == src.nodata, np.nan, profile)
        else:
            profile = np.where(profile < -100, np.nan, profile)

    bands_x   = np.arange(1, len(profile) + 1)
    c         = colors[i % len(colors)]
    label     = f'{filename[:25]}... (col={col_x}, row={row_y})'

    ax1.plot(bands_x, profile, label=label, color=c, linewidth=1.2, alpha=0.8)
    ax2.plot(bands_x, profile, color=c, linewidth=1.5)

    valid = profile[:285]
    valid = valid[~np.isnan(valid)]
    if len(valid) > 0:
        max_y_zoom = max(max_y_zoom, valid.max())
        min_y_zoom = min(min_y_zoom, valid.min())

ax1.set_title('Full spectral range (all bands)')
ax1.set_ylabel('Surface Reflectance')
ax1.grid(True, linestyle='--', alpha=0.7)
ax1.legend(loc='upper right', fontsize=9)

ax2.set_title('Zoom: EMIT reflectance bands 1–285')
ax2.set_xlabel('Band index')
ax2.set_ylabel('Surface Reflectance')
ax2.set_xlim(1, 285)
if max_y_zoom != -np.inf:
    margin = (max_y_zoom - min_y_zoom) * 0.1
    ax2.set_ylim(min_y_zoom - margin, max_y_zoom + margin)
ax2.grid(True, linestyle='--', alpha=0.7)

plt.suptitle('EMIT Spectral Profiles — Centre Pixel', y=1.02)
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'figures' / 'emit_spectral_profiles.png',
            dpi=150, bbox_inches='tight')
plt.show()


## 4. Single-Band Spatial Preview

Band 50 preview for up to 6 files.


In [ ]:
PREVIEW_BAND   = 50
max_to_plot    = min(len(tif_files), 6)
cols           = min(max_to_plot, 3)
rows           = math.ceil(max_to_plot / cols)

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
axes = np.array(axes).flatten()

for i in range(max_to_plot):
    with rasterio.open(tif_files[i]) as src:
        img, extent_bounds = reproject_raster_to_4326(src, band=PREVIEW_BAND)
        nodata = src.nodata
    img = np.where(img == nodata, np.nan, img) if nodata is not None           else np.where(img < -100, np.nan, img)

    ax = axes[i]
    im = ax.imshow(img, extent=extent_bounds, origin='upper', cmap='viridis')
    ax.set_aspect('equal')
    ax.set_title(f'{os.path.basename(tif_files[i])}\n(band {PREVIEW_BAND})', fontsize=9)
    format_map_axes(ax, fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

for j in range(max_to_plot, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'figures' / 'emit_band50_preview.png',
            dpi=150, bbox_inches='tight')
plt.show()


## 5. RGB Composite

True-colour composite using EMIT bands 54 / 35 / 19 (approx. 650 / 550 / 460 nm). 2–98 percentile contrast stretch applied per scene.


In [ ]:
R_BAND = 54
G_BAND = 35
B_BAND = 19

max_to_plot = min(len(tif_files), 6)
cols        = min(max_to_plot, 3)
rows        = math.ceil(max_to_plot / cols)

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
axes = np.array(axes).flatten()

for i in range(max_to_plot):
    with rasterio.open(tif_files[i]) as src:
        rgba, extent_bounds = reproject_rgb_composite_to_4326(src, [R_BAND, G_BAND, B_BAND])

    axes[i].imshow(rgba, extent=extent_bounds, origin='upper')
    axes[i].set_aspect('equal')
    axes[i].set_title(os.path.basename(tif_files[i]), fontsize=9, pad=8)
    format_map_axes(axes[i], fontsize=8)

for j in range(max_to_plot, len(axes)):
    axes[j].axis('off')

plt.suptitle(f'EMIT RGB Composite (bands {R_BAND}/{G_BAND}/{B_BAND}, 2-98% stretch)', y=1.02)
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'figures' / 'emit_rgb_composite.png',
            dpi=150, bbox_inches='tight')
plt.show()
